# SAINT fiel (`style="reference"`) — re-execução completa em DUAS T4

O SAINT foi reimplementado a partir do **código de referência** (`somepago/saint`, `RowColTransformer`,
estilo `colrow`): pré-normalização (`x + f(LN(x))`), FFN **GEGLU** (mult 4), MSA com `dim_head=16`, MISA
sobre a linha concatenada $(p+1)\cdot d$ com `dim_head=64`, FF2 na linha achatada, *embedding* por atributo
com FC+ReLU(100) e cabeça MLP(1000) sobre o `[CLS]`. Sem pré-treinamento contrastivo (declarado na tese).

**Por que refazer.** A primeira tentativa seguiu as Equações 1–2 do artigo, que descrevem **pós**-norma;
essa leitura não coincide com o código que gerou os resultados publicados, e sob lr 1e-3 sem aquecimento
ela colapsa para a classe majoritária (TWS 5/10 sementes contra 0/10 na pré-norma; TWC 10/10 contra 6/10;
AI4I 0,819 contra 0,858 — `results/saint_style_ablation.json`). Com o estilo de referência o SAINT volta ao
patamar dos números publicados, em vez de cair 0,067 no Tier 1.

**Fases:** Tier 1, Tier 2, Ablação D (N=5000), acréscimo do SAINT à Ablação A já rodada, Ablações B+C e as
duas linhas de SAINT na Tabela 19. As caras usam **as duas T4** (um processo por placa, metade das
sementes); a Tabela 19 roda em **uma** placa, por medir tempo e VRAM.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**. Estimativa: ≈5 a 6 h de sessão.
Tudo resumível. Os JSONs são gravados a cada execução em `<repo>/results/` e espelhados em
`/kaggle/working` a cada atualização de progresso (de onde se baixa); se a sessão cair, suba o parcial
como Dataset e aponte `RESUME_DIR` na célula 4.

**Concorrência** (mesmas verificações do notebook do entmax): um JSON por processo com escrita atômica;
todos os datasets já vêm em cache no clone, então nada é escrito em `data/raw/`; cada processo é preso a
uma placa e limitado a 2 threads de BLAS/OMP; `GridSearchCV` dos modelos de GPU roda com `n_jobs=1`.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2 (com 1 placa, troque run_phase_2gpu por run_phase)'

In [ ]:
# ── 2. Repositório + verificação do estilo do SAINT ──
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')
from src.models.ft_transformer_model import SAINTClassifier
_m = SAINTClassifier(n_features=5, d_model=8, n_heads=2, n_layers=1, dim_head=4)
_st = _m.stages[0]
assert _st.style == 'reference', f'estilo {_st.style!r} — o clone está velho, refaça o pull'
assert _st.ln3.normalized_shape == ((5 + 1) * 8,), 'FF2/LN3 deveriam operar na linha achatada (p+1)·d'
assert _st.misa.to_qkv.out_features == 3 * 2 * 64, 'MISA deveria usar dim_head=64'
print('SAINT no estilo de referência: pré-norma, GEGLU, MISA (p+1)·d com dim_head=64 — OK')

In [ ]:
# ── 3. Dependências e dados (tudo em cache no clone; nada é gerado nem baixado) ──
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
from pathlib import Path
USADOS = ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC',        # Tier 1
          'ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO',               # Tier 2 e Ablação D
          'TWS_2k','TWM_2k','TWC_2k',                                          # Ablação A
          'TWS_5f','TWM_5f','TWC_5f','MKE','MKM','MKH']                        # Ablações B e C
falt = [d for d in USADOS if not Path(f'data/raw/{d.upper()}.parquet').exists()]
assert not falt, f'sem cache (seriam gerados em paralelo): {falt} — refaça o pull'
from src.data.loaders import DatasetLoader
for ds in USADOS:
    X, y, _ = DatasetLoader.load(ds)
print(f'{len(USADOS)} datasets OK — todos em cache, nenhuma escrita concorrente possível')

In [ ]:
# ── 4. Configuração, progresso e paralelismo nas duas placas ──
import shutil, time, subprocess, collections
MODELS = ['SAINTColnorm']
MODELS_STR = ' '.join(MODELS)
ABL_A_MODELS = ['SAINTColnorm']     # só o SAINT: os outros cinco já rodaram na execução do entmax
ABL_A_STR = ' '.join(ABL_A_MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2   = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
SEEDS30 = ' '.join(map(str, range(30)))
SEEDS20 = ' '.join(map(str, range(20)))
OUT = {'tier1': 'results/saint_tier1.json',  'tier2': 'results/saint_tier2.json',
       'n5000': 'results/saint_n5000.json',   'ablA':  'results/saint_ablA.json',
       'ablBC': 'results/saint_ablBC.json',   'scal':  'results/saint_table19.json'}
nm = len(MODELS)
TOTAL = {'tier1': nm*10*30, 'tier2': nm*6*30, 'n5000': nm*6*30,
         'ablA': len(ABL_A_MODELS)*3*20, 'ablBC': nm*6*30, 'scal': 2*7}
Path('results').mkdir(exist_ok=True)

RESUME_DIR = None    # ex.: Path('/kaggle/input/entmax-parcial') para retomar sessão anterior
if RESUME_DIR:
    for k, v in OUT.items():
        src = Path(RESUME_DIR) / Path(v).name
        if src.exists(): shutil.copy(src, v); print('restaurado', v)

def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

def _mirror(*paths):
    """Copia os JSONs em curso para /kaggle/working (raiz do Output).

    Os runners gravam em <repo>/results/ a cada execução concluída (append + escrita
    atômica), mas o painel Output do Kaggle mostra /kaggle/working: sem esta cópia
    periódica, uma sessão interrompida no meio de uma fase deixaria os dados só no
    diretório aninhado do clone. Espelhar a cada atualização de progresso torna os
    parciais sempre baixáveis (e reaproveitáveis via RESUME_DIR)."""
    for p in paths:
        try:
            if Path(p).exists(): shutil.copy(p, '/kaggle/working/' + Path(p).name)
        except Exception as e:
            print('  (aviso: falha ao espelhar', p, e, ')')

def _count(path):
    try: r = json.load(open(path))
    except Exception: return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok: return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = f"último: {x.get('dataset', 'N=' + str(x.get('n')))}"
    info += f"/seed{x['seed']}" if 'seed' in x else ''
    info += f" F1={f1:.3f}" if isinstance(f1, (int, float)) else ''
    return len(ok), info

def run_phase(key, cmd, every=60, heartbeat=900):
    """Uma GPU."""
    out, total = OUT[key], TOTAL[key]; log = f'/kaggle/working/{key}.log'
    t0 = time.time(); last_n, last_print = -1, 0.0
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT)
        while True:
            rc = p.poll(); n, info = _count(out); now = time.time()
            if n != last_n or rc is not None or now - last_print > heartbeat:
                el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
                print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
                _mirror(out)                     # parcial sempre baixável
                last_n, last_print = n, now
            if rc is not None: break
            time.sleep(every)
    print(f'[{key}] terminou (exit={rc}) — log: {log}'); save(key)
    if rc != 0: raise RuntimeError(f'{key}: exit {rc}\n' + ''.join(open(log).readlines()[-15:]))

def run_phase_2gpu(key, cmd_fmt, seeds=range(30), every=60, heartbeat=900):
    """Duas T4: um processo por placa, metade das sementes cada, shards separados.

    NÃO usar DataParallel/DDP: o FT-CUR e o SAINT fazem atenção inter-instâncias
    DENTRO do lote, então dividir o lote entre placas mudaria o modelo. Aqui o
    paralelismo é por experimento (sementes), que não altera nada.
    Dois processos gravando o mesmo JSON se sobrescrevem → um arquivo por placa.
    """
    seeds = list(seeds); half = len(seeds)//2
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs, rcs = [], [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        lf = open(log, 'w')
        # CUDA_VISIBLE_DEVICES prende o processo a uma placa; os limites de thread evitam
        # que os dois processos disputem os poucos vCPUs do Kaggle (cada um abriria tantas
        # threads de BLAS/OMP quantos núcleos houver, e o oversubscription chega a dobrar o
        # tempo do pré-processamento).
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                   NUMEXPR_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min', flush=True)
            _mirror(*shards)                     # parciais sempre baixáveis
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros'); save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs): raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')

print('config OK — modelos:', MODELS)
for k, v in OUT.items(): print(f'  {k:<6} -> {v}  (total {TOTAL[k]})')

In [ ]:
# ── 5. Tier 1 (10 datasets × 30 sementes) — duas placas ──
run_phase_2gpu('tier1', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER1 + " --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 6. Tier 2 (N=2000) — duas placas ──
run_phase_2gpu('tier2', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 2000 --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 7. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 NOVO) ──
recs = [r for r in json.load(open(OUT['tier2'])) if r.get('status') == 'ok' and r.get('best_params')]
by = collections.defaultdict(list)
for r in recs: by[(r['variant'], r['dataset'])].append(tuple(sorted(r['best_params'].items())))
cfg_fixed = json.load(open('config/tier2_fixed_params.json'))
for (v, d), vals in sorted(by.items()):
    mode, cnt = collections.Counter(vals).most_common(1)[0]
    cfg_fixed.setdefault(v, {})[d] = dict(mode)
    print(f'{v:<16} {d:<9} {dict(mode)}  [moda {cnt}/{len(vals)}]')
Path('config/tier2_fixed_params_saint.json').write_text(json.dumps(cfg_fixed, indent=2, sort_keys=True))
shutil.copy('config/tier2_fixed_params_saint.json', '/kaggle/working/tier2_fixed_params_saint.json')
run_phase_2gpu('n5000', "python -u scripts/run_tier2_fixedparams.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 5000 --config config/tier2_fixed_params_saint.json"
               " --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 8. Ablação A — acrescenta o SAINT (transferência do Tier 1 recém-rodado) ──
# Os outros cinco Transformers já foram rodados na execução do entmax; aqui só o SAINT.
# O merge abaixo é local à sessão: põe os best_params novos do SAINT no tier1 que o script lê.
!python scripts/merge_rerun_results.py --target results/tier1_gridcv.json --source {OUT['tier1']} --variants SAINTColnorm --tag saint
run_phase_2gpu('ablA', "python -u scripts/run_ablation_a_scaling.py --models " + ABL_A_STR +
               " --tier1 results/tier1_gridcv.json --seeds {seeds} --output {output}", seeds=range(20))
print(collections.Counter((r['variant'], r.get('protocol')) for r in json.load(open(OUT['ablA'])) if r.get('status') == 'ok'))

In [ ]:
# ── 9. Ablações B + C — duas placas ──
run_phase_2gpu('ablBC', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets TWS_5f TWM_5f TWC_5f MKE MKM MKH --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 10. Tabela 19 — UMA placa (mede tempo e VRAM; nada concorrente) ──
# SAINT_fullbatch esgota memória em N grande: é o comportamento publicado (a variante
# existe para mostrar o custo O(N²) da atenção inter-instâncias densa e global).
run_phase('scal', f"CUDA_VISIBLE_DEVICES=0 python -u scripts/run_table19_benchmark.py --variants SAINT_minibatch,SAINT_fullbatch --repeats 3 --output {OUT['scal']}")

In [ ]:
# ── 11. Resumo e comparação com os números publicados (versão só-CLS) ──
PUB_T1 = {'AI4I':0.808,'AUS':0.857,'BCW':0.945,'GCR':0.669,'HAB':0.534,
          'PID':0.712,'TWC':0.460,'TWM':0.942,'TWS':0.641,'VCP':0.815}
for key in ['tier1', 'tier2', 'n5000', 'ablA', 'ablBC']:
    p = Path(OUT[key])
    if not p.exists(): print(key, 'ausente'); continue
    ok = [r for r in json.load(open(p)) if r.get('status') == 'ok']
    f1 = collections.defaultdict(list)
    for r in ok: f1[r['dataset']].append(r['test_f1_macro'])
    print(f'\n=== {key}: {len(ok)}/{TOTAL[key]} ===')
    for d, vals in sorted(f1.items()):
        ref = f"   publicado(só-CLS) {PUB_T1[d]:.3f}" if key == 'tier1' and d in PUB_T1 else ''
        print(f'  {d:<9} {st.mean(vals):.4f} ± {st.pstdev(vals):.3f}  n={len(vals)}{ref}')
    if key == 'tier1':
        todos = [v for vals in f1.values() for v in vals]
        print(f'  MÉDIA    {st.mean(todos):.4f}   (publicado só-CLS 0,7384; pós-norma descartada 0,6714)')
print('\nBaixe de /kaggle/working: saint_{tier1,tier2,n5000,ablA,ablBC,table19}.json')